# 1.2 Experimentos de ingeniería de datos: corridas en Colab

Entrena las corridas definidas en `oe1/1.2_ingenieria_datos.ipynb` (sección 7). Cada corrida usa el protocolo de `comun.correr`: YOLOv11s preentrenado en COCO, SGD (`lr0 = 0.01`), lote 8, aumento en línea por defecto de Ultralytics y 50 épocas sin parada temprana (menos con balanceo, mismo número de iteraciones). Se evalúa en su pliegue de validación y añade una fila a `oe1/resultados/experimentos.csv`.

| Fase | Corridas | Tiempo aproximado en una A100 |
| :--- | :---: | :--- |
| `factorial`: 12 configuraciones × 4 pliegues × 3 semillas | 144 | ~3.3 h por semilla |
| `complementarios`: etiquetas originales y sin aumento, sobre la configuración elegida | 24 | ~0.6 h por semilla |

**Uso:**
1. Subir `oe1_nube.zip` a la raíz de Google Drive y abrir este notebook con una **A100**.
2. Elegir `FASE` y las semillas de la sesión (`SEMILLAS_SESION`). Se recomienda **una semilla por sesión**.
3. Ejecutar todo. Al terminar, la sesión descarga `oe1_runs_<fase>_s<semillas>.zip` con las corridas y el registro.
4. **Varias sesiones:** cada una entrena solo sus semillas, así que pueden correr a la vez (una copia del notebook por sesión) y no hace falta subir a Drive los zips descargados. Solo si se repite una sesión con semillas ya hechas, su zip se sube a la raíz de Drive para que esas corridas se omitan.
5. En Drive debe haber **un solo** `oe1_nube*.zip`: al regenerarlo, se reemplaza el anterior.

En local, `oe1/1.2_experimentos.ipynb` incorpora las descargas y analiza los resultados. Los `complementarios` necesitan `resultados/decision.json` en el paquete: se regenera el zip después de la decisión.

In [ ]:
FASE = 'factorial'        # 'factorial' o 'complementarios'
SEMILLAS_SESION = (13,)   # semillas de esta sesión: (13,), (29,), (51,) o varias, p. ej. (13, 29, 51)

In [ ]:
# --- Entorno: local o Colab ---
import os, sys, glob, json, shutil, subprocess, time, zipfile
from pathlib import Path

os.environ['YOLO_VERBOSE'] = 'False'   # sin barras de progreso: la interfaz de Colab se congela con tanta salida
PLATAFORMA = 'colab' if 'COLAB_RELEASE_TAG' in os.environ else 'local'
if PLATAFORMA == 'local':   # este notebook vive en nube/; el código del OE1, en oe1/
    RAIZ = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'oe1' / 'comun.py').exists()) / 'oe1'
else:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.174', 'albumentations==2.0.8'], check=True)
    from google.colab import drive
    drive.mount('/content/drive')   # el paquete está en Google Drive, que solo se lee
    base, origen = Path('/content'), '/content/drive/MyDrive'
    if not glob.glob(str(base / '*' / 'oe1' / 'comun.py')):
        # primero la raíz de Drive; recorrerlo entero es lento, solo si no está ahí
        paquete = glob.glob(f'{origen}/oe1_nube*.zip') or glob.glob(f'{origen}/**/oe1_nube*.zip', recursive=True)
        assert paquete, f"No se encontró oe1_nube.zip en Drive. Zips en la raíz: {glob.glob(f'{origen}/*.zip')}"
        zipfile.ZipFile(paquete[0]).extractall(base)
    RAIZ = Path(glob.glob(str(base / '*' / 'oe1' / 'comun.py'))[0]).parent
    # Corridas de sesiones anteriores: los zips descargados, subidos a la raíz de Drive o en esta máquina
    registro_previo = []
    for previo in sorted(glob.glob(f'{origen}/oe1_runs*.zip') + glob.glob(str(base / 'oe1_runs*.zip'))):
        with zipfile.ZipFile(previo) as z:
            for n in z.namelist():
                if n.startswith('runs/'):
                    z.extract(n, RAIZ.parent)
                elif n.endswith('experimentos.csv'):
                    registro_previo.append(z.read(n))
        print('Restaurado:', previo)
os.chdir(RAIZ); sys.path.insert(0, str(RAIZ))

import pandas as pd
import comun as C
import torch

part = json.loads((RAIZ / 'resultados' / 'particion.json').read_text())
REGISTRO = RAIZ / 'resultados' / 'experimentos.csv'
if PLATAFORMA == 'colab' and registro_previo:   # une los registros del paquete y de las sesiones anteriores
    import io
    tablas = [pd.read_csv(REGISTRO)] if REGISTRO.exists() else []
    tablas += [pd.read_csv(io.BytesIO(b)) for b in registro_previo]
    pd.concat(tablas, ignore_index=True).drop_duplicates('id', keep='last').to_csv(REGISTRO, index=False)
assert torch.cuda.is_available(), 'No hay GPU: Entorno de ejecución > Cambiar tipo de entorno > A100'
GPU = torch.cuda.get_device_name(0)
print('Plataforma:', PLATAFORMA, '| GPU:', GPU, '| fase:', FASE, '| semillas:', SEMILLAS_SESION)
if PLATAFORMA == 'colab' and 'A100' not in GPU:
    print('AVISO: todas las corridas deben usar la misma GPU (A100).')

## 1. Plan de la sesión

Corridas de la fase y las semillas elegidas que aún no están en el registro. En local no se entrena: la GPU de 6 GB no admite 1280 px con lote 8 y todas las corridas deben usar la misma GPU.

In [ ]:
if FASE == 'factorial':
    experimentos = C.matriz_experimentos(part, SEMILLAS_SESION)
elif FASE == 'complementarios':
    decision = json.loads((RAIZ / 'resultados' / 'decision.json').read_text())
    experimentos = C.experimentos_complementarios(part, decision, SEMILLAS_SESION)
else:
    raise ValueError(FASE)
hechos = set(pd.read_csv(REGISTRO).id) if REGISTRO.exists() else set()
faltan = [e for e in experimentos if e['id'] not in hechos]
pendientes = faltan if PLATAFORMA == 'colab' else []
print(f'{len(experimentos)} corridas de la fase con estas semillas; {len(experimentos) - len(faltan)} ya registradas; '
      f'{len(pendientes)} se ejecutan en esta sesión')
pd.DataFrame(faltan).groupby(['semilla', 'resolucion']).size().rename('pendientes').to_frame() if faltan else None

## 2. Corridas

Cada corrida se registra al terminar. Su carpeta `runs/oe1/<id>/` conserva los resultados de Ultralytics (curvas, matrices de confusión, argumentos) y `best.pt` como único peso.

In [ ]:
t0 = time.time()
for i, exp in enumerate(pendientes, 1):
    print(f"[{i}/{len(pendientes)}] {exp['id']}")
    f = C.correr(exp, part, C.DATOS, C.TRABAJO, C.RUNS, REGISTRO)   # libera la VRAM al terminar
    restante = (time.time() - t0) / i * (len(pendientes) - i) / 3600
    print(f"   mAP50-95 {f['mAP50_95']:.4f} | épocas {f['epocas']} | {f['minutos']} min | VRAM máx. {f['vram_gb']} GB | faltan ~{restante:.1f} h")

## 3. Descarga

Empaqueta el registro y las carpetas de las corridas de esta fase y semillas en `oe1_runs_<fase>_s<semillas>.zip` y lo descarga.

In [ ]:
if PLATAFORMA == 'colab' and REGISTRO.exists():
    destino = Path('/content') / f"oe1_runs_{FASE}_s{'_'.join(map(str, SEMILLAS_SESION))}.zip"
    ids = {e['id'] for e in experimentos}
    with zipfile.ZipFile(destino, 'w', zipfile.ZIP_DEFLATED) as z:
        z.write(REGISTRO, REGISTRO.relative_to(C.REPO))
        for f in C.RUNS.rglob('*'):
            if f.is_file() and f.relative_to(C.RUNS).parts[0] in ids:   # los .pt ya están comprimidos
                z.write(f, f.relative_to(C.REPO), compress_type=zipfile.ZIP_STORED if f.suffix == '.pt' else zipfile.ZIP_DEFLATED)
    print(f'{destino} ({destino.stat().st_size / 1e6:.0f} MB)')
    from google.colab import files
    files.download(str(destino))